In [1]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
import clip
from PIL import Image

from groundingdino.util.inference import load_model, predict, load_image
from segment_anything import sam_model_registry, SamPredictor

In [2]:
IMAGE_DIR = "./data/damaged-sign3/labels"
LABEL_DIR = "./data/damaged-sign3/labels"

os.makedirs(LABEL_DIR, exist_ok=True)

GROUNDING_MODEL_CONFIG = "./GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
GROUNDING_MODEL_CHECKPOINT = "./groundingdino_swint_ogc.pth"

SAM_CHECKPOINT = "./sam_vit_b_01ec64.pth"
DEVICE = "cpu"

In [3]:
grounding_model = load_model(
    GROUNDING_MODEL_CONFIG,
    GROUNDING_MODEL_CHECKPOINT
)
grounding_model = grounding_model.to("cpu")

final text_encoder_type: bert-base-uncased


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
sam = sam_model_registry["vit_b"](checkpoint="./sam_vit_b_01ec64.pth")
sam.to(device=DEVICE)
sam_predictor = SamPredictor(sam)

In [5]:
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)

In [9]:
TEXT_PROMPT = "road sign"
BOX_THRESHOLD = 0.3
TEXT_THRESHOLD = 0.25

CLASS_NAMES = ["regulatory", "warning", "information", "guide"]

TEXT_PROMPTS_CLIP = {
    "regulatory": [
        "a white rectangular traffic sign with a prominent red circle and black numbers inside",
        "a regulatory road sign with a thick red border and black text",
        "a stop sign with an octagonal shape and red background",
        "an inverted triangle give way sign with a thick red border",
        "a white rectangular parking sign with green or red text and symbols",
        "a white sign with black text indicating a bus lane or specialized vehicle lane",
        "a regulatory traffic sign displaying a mandatory instruction or legal restriction"
    ],

    "warning": [
        "a yellow diamond-shaped traffic sign with a black symbol in the center",
        "a yellow rectangular plate with black text indicating an advisory speed in km/h",
        "a warning sign with a yellow background and a black arrow or curve symbol",
        "a yellow hazard sign warning of road conditions, pedestrians, or animals",
        "a black and white chevron or hazard marker sign with arrow patterns",
        "a yellow sign with black text saying No Through Road or Speed Hump",
        "a yellow diamond sign warning of an upcoming intersection or regulatory sign"
    ],

    "information": [
        "a blue rectangular road sign with white icons for hospital, fuel, or food services",
        "a blue information sign showing a soccer ball, sporting facility, or community center",
        "a brown rectangular tourist sign indicating a historical site or national park",
        "a blue sign with a white 'P' or 'i' symbol for parking or information",
        "a blue or brown sign with white text describing a point of interest or facility"
    ],

    "guide": [
        "a large green directional sign with white destination names and arrows",
        "a green highway sign with alpha-numeric route shields like M1, A1, or B20",
        "a green freeway exit sign with distance markers in meters or kilometers",
        "a white rectangular street name sign at an intersection",
        "a white or green sign with an arrow pointing toward a suburb or city center",
        "a green overhead gantry sign displaying road names and lane directions"
    ]
}

In [10]:
# # Precompute text features once (very important for speed)
# text_tokens = clip.tokenize(TEXT_PROMPTS_CLIP).to(DEVICE)
# with torch.no_grad():
#     text_features = clip_model.encode_text(text_tokens)
#     text_features /= text_features.norm(dim=-1, keepdim=True)   # normalize

# texts = []
# text_class_map = []

# for class_id, class_name in enumerate(CLASS_NAMES):
#     for prompt in TEXT_PROMPTS_CLIP[class_name]:
#         texts.append(prompt)
#         text_class_map.append(class_id)

# text_tokens = clip.tokenize(texts).to(DEVICE)

# with torch.no_grad():
#     text_features = clip_model.encode_text(text_tokens)
#     text_features /= text_features.norm(dim=-1, keepdim=True)

class_features = []

for class_name in CLASS_NAMES:
    prompts = TEXT_PROMPTS_CLIP[class_name]
    tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        features = clip_model.encode_text(tokens)
        features /= features.norm(dim=-1, keepdim=True)
        features = features.mean(dim=0)
        features /= features.norm()

    class_features.append(features)

text_features = torch.stack(class_features)

In [13]:
for image_name in tqdm(os.listdir(IMAGE_DIR)):

    if not image_name.lower().endswith((".jpg")):
        continue

    label_path = os.path.join(
        LABEL_DIR,
        image_name.replace(".jpg", ".txt").replace(".png", ".txt")
    )

    # Skip images already processed
    if os.path.exists(label_path):
        continue

    image_path = os.path.join(IMAGE_DIR, image_name)
    image_source, image = load_image(image_path)
    h, w, _ = image_source.shape

    # GroundingDINO detection
    boxes, logits, phrases = predict(
        model=grounding_model,
        image=image,
        caption=TEXT_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        device=DEVICE
    )

    sam_predictor.set_image(image_source)

    label_path = os.path.join(
        LABEL_DIR,
        image_name.replace(".jpg", ".txt").replace(".png", ".txt")
    )

    with open(label_path, "w") as f:

        for box in boxes:

            cx, cy, bw, bh = box

            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            x2 = (cx + bw / 2) * w
            y2 = (cy + bh / 2) * h

            input_box = np.array([x1, y1, x2, y2]).astype(int)

            masks, scores, logits = sam_predictor.predict(
                box=input_box,
                multimask_output=False
            )

            mask = masks[0]

            # convert mask → bounding box
            ys, xs = np.where(mask)

            if len(xs) == 0 or len(ys) == 0:
                continue

            x_min, x_max = xs.min(), xs.max()
            y_min, y_max = ys.min(), ys.max()

            # crop sign with bounding box
            pad = 7
            x_min_crop = max(0, x_min - pad)
            x_max_crop = min(w, x_max + pad)
            y_min_crop = max(0, y_min - pad)
            y_max_crop = min(h, y_max + pad)

            cropped = image_source[y_min_crop:y_max_crop, x_min_crop:x_max_crop]
            if cropped.size == 0:
                continue

            # Convert to PIL for CLIP
            cropped_pil = Image.fromarray(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))

            # CLIP classification
            cropped_pre = clip_preprocess(cropped_pil).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                image_features = clip_model.encode_image(cropped_pre)
                image_features /= image_features.norm(dim=-1, keepdim=True)

                # similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
                # best_idx = similarity.argmax().item()
                # confidence = similarity[0, best_idx].item()

                # similarity = image_features @ text_features.T
                # best_prompt = similarity.argmax().item()
                # best_idx = text_class_map[best_prompt]
                # confidence = similarity[0, best_prompt].item()

                similarity = image_features @ text_features.T
                best_idx = similarity.argmax().item()
                confidence = similarity[0, best_idx].item()
            
            # if confidence < 0.4: best_idx = 4  # unknown / info

            # convert to YOLO format
            x_center = ((x_min + x_max) / 2) / w
            y_center = ((y_min + y_max) / 2) / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            class_id = best_idx
            # class_id = 0  # road_sign
            print(f"{image_name} → {CLASS_NAMES[best_idx]} ({confidence:.3f})")
            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

  8%|▊         | 1/12 [00:14<02:44, 14.93s/it]

91343852333183574241875713112609539407478883231-25.jpg → regulatory (0.268)
91343852333185646637681399511132643006571516199-25.jpg → information (0.255)


 17%|█▋        | 2/12 [00:29<02:28, 14.82s/it]

91343852333185646637681399511132643006571516199-25.jpg → information (0.216)


 25%|██▌       | 3/12 [00:44<02:12, 14.74s/it]

...

91343852333188861731371508829378246470926802229-25.jpg → information (0.242)
91343852333198849297910939034286156041438995569-0.jpg → guide (0.313)
91343852333198849297910939034286156041438995569-0.jpg → warning (0.322)
91343852333198849297910939034286156041438995569-0.jpg → information (0.254)


 83%|████████▎ | 10/12 [02:26<00:29, 14.70s/it]

91343852333217397071269222930319934049355990653-0.jpg → regulatory (0.295)
91343852333217397071269222930319934049355990653-0.jpg → regulatory (0.325)
91343852333217397071269222930319934049355990653-0.jpg → information (0.269)
91343852333217397071269222930319934049355990653-0.jpg → warning (0.194)


100%|██████████| 12/12 [02:41<00:00, 13.50s/it]

91343852333217397071269222930319934049355990653-0.jpg → warning (0.311)
